<a href="https://colab.research.google.com/github/BryanHinostroza/lab14-bh/blob/develop/GLAB14_MDD_HINOSTROZA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SEMANA 14: CONCEPTOS PRELIMINARES. MULTICLASIFICADORES**


## INTEGRANTES:

- GUTIERREZ GARCIA, Angela Belen
- HINOSTROZA MARTINEZ, Bryan Jesus



Tomando la información disponible en el repositorio UCI Machine Learning
https://archive.ics.uci.edu/ml/datasets/Productivity+Prediction+of+Garment+Employees, pero eliminando previamente a la variable ‘date’ y tomando a la variable ‘actual_productivity’ como
variable predictora, haga lo siguiente:

### EJERCICIO 🅐:


Realice el preprocesamiento de la información que incluya el análisis de datos faltantes y
tratamiento de outliers a nivel univariado y multivariado. Además, convierta las variables
categóricas a dummies y aplique un escalamiento a las variables numéricas.

In [ ]:
!pip install ucimlrepo

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Cargamos la base de datos desde UCI ML Repository
data = fetch_ucirepo(id=597)  # Productivity Prediction of Garment Employees
df = pd.concat([data.data.features, data.data.targets], axis=1)

# 2. Eliminamos la columna 'date' ya que no aporta valor predictivo
df = df.drop(columns=['date'])

# 3. Análisis de valores faltantes
print("Valores nulos por columna:")
print(df.isnull().sum())

# Imputamos los valores faltantes con la mediana de cada columna
df = df.fillna(df.median(numeric_only=True))

# 4. Identificación de variables categóricas y numéricas
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=np.number).drop(columns='actual_productivity').columns.tolist()

# 5. Tratamiento de outliers univariados (por IQR)
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower, upper)

# 6. Codificación de variables categóricas
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 7. Escalado de variables numéricas
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df.drop(columns=['actual_productivity']))
df_scaled = pd.DataFrame(scaled_features, columns=df.drop(columns=['actual_productivity']).columns)
df_scaled['actual_productivity'] = df['actual_productivity']


Valores nulos por columna:
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


### EJERCICIO 🅑:  

Separe los datos en entrenamiento (80%) y prueba (20%) y, realice un ensamblaje utilizando las técnicas de voting, bagging, boosting y stacking para los modelos k-NN, SVM, regresión lineal, árbol de clasificación y Random Forest definiendo distintos hiperparámetros para cada
modelo.

In [ ]:
# Separar X (features) e y (target)
X = df_scaled.drop(columns='actual_productivity')
y = df_scaled['actual_productivity']

# División 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modelos base
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=0)
knn = KNeighborsRegressor(n_neighbors=10)
svr = SVR(kernel='rbf', C=1.0)
dt = DecisionTreeRegressor(max_depth=5, random_state=0)

# Modelos de ensamblaje
models = {
    "Voting": VotingRegressor([("lr", lr), ("rf", rf), ("knn", knn)]),
    # Cambiamos 'base_estimator' a 'estimator' para versiones recientes de scikit-learn
    "Bagging": BaggingRegressor(estimator=dt, n_estimators=50, random_state=0),
    "Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0),
    "Stacking": StackingRegressor(estimators=[("rf", rf), ("knn", knn), ("svr", svr)], final_estimator=lr)
}


### EJERCICIO 🅒:  

Utilice las métricas error cuadrático medio, raíz del error cuadrático medio, error absoluto
medio y coeficiente de determinación para elegir el mejor modelo que prediga a la variable ‘actual_productivity’, justificando su respuesta.

In [ ]:
# Evaluamos cada modelo
resultados = []

for nombre, modelo in models.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    resultados.append({
        "Modelo": nombre,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

# Mostrar resultados ordenados por R2
resultados_df = pd.DataFrame(resultados).sort_values(by="R2", ascending=False).reset_index(drop=True)
print(resultados_df)

     Modelo       MSE      RMSE       MAE        R2
0   Bagging  0.013696  0.117028  0.078572  0.484209
1  Stacking  0.013933  0.118039  0.078537  0.475255
2  Boosting  0.015135  0.123024  0.081268  0.430002
3    Voting  0.015497  0.124485  0.085576  0.416378


### Justificación del mejor modelo

In [ ]:
mejor_modelo = resultados_df.iloc[0]
print(f"""
El modelo con mejor rendimiento es: {mejor_modelo['Modelo']}
Con un:
R2: {mejor_modelo['R2']:.4f}
RMSE: {mejor_modelo['RMSE']:.4f}
MAE: {mejor_modelo['MAE']:.4f}

Justificación:
Se selecciona este modelo porque tiene el mayor coeficiente de determinación (R2), lo cual indica que explica mejor la variabilidad
en la productividad real de los trabajadores
""")



El modelo con mejor rendimiento es: Bagging
Con un: 
R2: 0.4842
RMSE: 0.1170
MAE: 0.0786

Justificación:
Se selecciona este modelo porque tiene el mayor coeficiente de determinación (R2), lo cual indica que explica mejor la variabilidad
en la productividad real de los trabajadores

